# Task 1.3 — What the Paper Claims to Improve


---
## Baseline 1: Standard SVM with Non-Symmetric Pairwise Kernels (K_D / K_T)

### Baseline Description
The most direct baseline is using a plain **Direct Kernel** ($K_D$) or **Tensor Kernel** ($K_T$) without linearisation:
- $K_D((a,b),(c,d)) = k(a,c) + k(b,d)$
- $K_T((a,b),(c,d)) = k(a,c) \cdot k(b,d)$

These are natural first choices for lifting a standard SVM to work on pairs.

### Limitation of Baseline 1
These kernels are **not symmetric**: $K_D((a,b),(c,d)) \neq K_D((b,a),(c,d))$ in general. This means the trained model will give **different predictions** for the same pair depending on which input is presented first. In face verification or any identity-matching task, this is logically incorrect — the same two people should yield the same answer regardless of order. This inconsistency can reduce accuracy and undermines the interpretability of the model.

### How the Paper's Method Overcomes It
The paper introduces the **linearised (balanced) versions** $K_{DL}$ and $K_{TL}$ which are symmetric by construction (Section 3, Theorem 2). By adding the cross-terms:
$$K_{DL}((a,b),(c,d)) = k(a,c) + k(b,d) + k(a,d) + k(b,c)$$
the kernel becomes invariant to swapping inputs. The paper proves this produces the same decision function as training with a fully symmetric dataset (Theorem 3), while being more efficient.

### When Balanced Kernel Would NOT Win
If the task is **inherently asymmetric** (e.g., document-to-template matching, where one input is a fixed reference), breaking the ordering by symmetrising the kernel would lose information. In such cases, $K_D$ without linearisation would actually be more appropriate and would produce better predictions than $K_{DL}$.
### Note on Evaluation Metric

The paper's primary metric is **Equal Error Rate (EER)** — the point where False Acceptance Rate equals False Rejection Rate. Lower EER = better. The best result in the paper is EER = 0.0947 (roughly 90.5% verification accuracy) on LFW using fused SIFT+LBP+TPLBP features with $K_{DL}$. The baseline (SVM on concatenated features) achieves EER = 0.0993. So the actual improvement is about 0.5% in EER, which sounds small but is significant in biometric verification.


---
## Baseline 2: Symmetric Training Set with Standard Pairwise Kernel

### Baseline Description
An alternative way to enforce symmetry is to **augment the training set**: for every pair $(a,b)$, also add $(b,a)$ with the same label. This symmetric training set ($\hat{S}$) is then used with the plain $K_D$ kernel. This is a straightforward, brute-force way to achieve symmetry.

### Limitation of Baseline 2
This approach **doubles the training set size**. For datasets with millions of pairs (as in LFW), this leads to:
1. $2\times$ the memory consumption for storing pairs
2. $4\times$ the computational cost for SVM training (since SVM training scales roughly as $O(n^2)$ to $O(n^3)$)

This makes the approach infeasible for large-scale problems. Additionally, the redundant pairs don't add new information — they just impose symmetry artificially by duplication.

### How the Paper's Method Overcomes It
The key result in **Theorem 3** (Section 3) proves that:
> Using $K_{DL}$ with the original (non-symmetric) training set $S$ yields **exactly the same** decision function as using $K_D$ with the symmetric training set $\hat{S}$.

So the balanced kernel achieves the same outcome with half the data. The paper also shows in Section 4 that the extra kernel evaluations in $K_{DL}$ can be computed by caching $k$ values, making the overhead minimal.

### When Symmetric Training Set Would NOT Lose to Balanced Kernel
If the **dataset is very small** (e.g., fewer than 100 pairs), the $2\times$ training overhead is negligible. In that setting, using a symmetric training set is simpler to implement and understand, with no meaningful speed disadvantage. The scalability benefit of balanced kernels only pays off at large scale.

The table above shows why doubling the training set is prohibitive at scale — even at 1,000 identities, the symmetric set grows to millions of pairs. This is exactly the motivation behind the balanced kernel approach.